# Bulk invoice and contract review

Mount a folder of invoices and contracts alongside a reusable accounts-payable policy skill. A `gpt-5.6-luna` agent delegates each document to a specialist subagent, applies the discovered skill, writes individual reports plus a consolidated summary, and leaves every approval to a person.

```mermaid
sequenceDiagram
    participant Person
    participant App as Review command
    participant Agent as Agents API
    participant Reviewers as Specialist subagents
    participant Policy as Mounted review skill
    participant Workspace as Mounted input/output folders
    Person->>App: Submit a folder of invoices and contracts
    App->>Agent: Create one multi-agent session
    App->>Workspace: Mount input and policy read-only; output read-write
    Agent->>Reviewers: Delegate one document to each specialist
    Reviewers->>Policy: Discover and apply expense-review-policy
    Reviewers->>Workspace: Write individual JSON reports
    Agent->>Workspace: Write the consolidated summary
    App-->>Person: Write findings for human review
```


## Agents API capabilities

Sandbox, Multi-agent, Skills, Workspace files, Turn history, Streaming.

### Review documents in parallel

Enable multi-agent execution so the coordinator delegates individual invoices and contracts to specialist subagents instead of reviewing a batch sequentially.

### Discover reusable policy skills

Mount an accounts-payable policy into the sandbox and register its capability root. Every specialist discovers the same skill instead of duplicating policy rules in application prompts.

### Work directly with mounted files

Input documents are mounted read-only, while specialists write reports to a separate output directory that remains available after the sandbox exits.

### Produce durable batch artifacts

Each document gets a machine-readable JSON report, and the coordinating agent produces a consolidated summary for the complete batch.

### Keep approval with a person

The agent identifies risks and recommends a decision, but your application retains authority over payment approval and contract acceptance.


## Application flow

1. Document folder.
2. Mounted policy skill.
3. Agent coordinator.
4. Specialist subagents.
5. Review artifacts.
6. Human approvals.


## What you need

- Python 3.14+ and `uv`.
- A sandbox: self-hosted Docker or a [third-party provider](https://developers.openai.com/api/docs/guides/agents-api/environments/self-hosted#sandbox-providers).
- An OpenAI API key and a separate restricted executor key.


## 1. Set up the workspace

From the repository root:

```bash
cp examples/agents_api/apps/document_review/.env.example examples/agents_api/apps/document_review/.env
docker build -t agent-api-sandbox:latest examples/agents_api/sandboxes/application_managed/docker
```

Set `OPENAI_API_KEY` and `OPENAI_EXECUTOR_API_KEY` in `examples/agents_api/apps/document_review/.env`. Use keys with the same owner, organization, and project. Only the executor key enters the sandbox. It needs `api.agents.environments.connect` and IP restrictions that allow the sandbox's outbound network. The application loads this file automatically.

To create an executor key with the required permission, open [Agents > Environments > Keys](https://platform.openai.com/agents?tab=environments&environment_view=keys) and select **Create**.

You can replace local Docker with any compatible [sandbox provider](https://developers.openai.com/api/docs/guides/agents-api/environments/self-hosted#sandbox-providers).


## 2. Review the document batch

```bash
uv run examples/agents_api/apps/document_review/main.py \
  --input examples/agents_api/apps/document_review/sample_documents \
  --output ./review-output
```

The input folder contains an invoice with a $900 overcharge and a contract with automatic renewal, unilateral price increases, unlimited liability, and unrestricted customer-data sharing. Specialist subagents apply the mounted policy to each document and write:

```text
review-output/
  contract.json
  invoice.json
  summary.json
  review-activity.json
```

The source folder and policy skill are mounted read-only; reports are written to `/workspace/output` and remain available on the host after the sandbox exits. With Docker Desktop, keep input and output directories under your home directory; system temporary directories may not be shared.

The command logs the session, sandbox, mounted directories, specialist subagents, and generated artifacts as the review progresses.

Invoice reports include extracted line items and a calculation. The application
checks the arithmetic before returning a report. A person must still verify that
the extracted amounts match the source document and review the recommendation.

Use an empty output folder for each run and unique document stems, such as `invoice-104.txt` and `contract-208.txt`. The names `summary` and `review-activity` are reserved.


## Inspect retained command activity

Before deleting the session, the app exports retained commands from the coordinator
and each specialist to `review-activity.json`. A `null` subagent ID identifies the
coordinator. Specialist commands come from their own item histories:

The corresponding implementation is included in the Python code cells below.

These records come from retained API history, not model-written reviewer names.
They are not a complete security audit. Protect this file like the reports:
commands can contain document content.


## How the policy skill works

The included policy lives at:

```text
skills/expense-review-policy/SKILL.md
```

It is mounted at `/workspace/skills/expense-review-policy/SKILL.md` and discovered through the session's capability root:

The corresponding implementation is included in the Python code cells below.

Each specialist applies `$expense-review-policy` and includes its `policy_id` and `decision` in the generated report. Replace `SKILL.md` with your own review policy without rebuilding the sandbox image.


## Make it yours

Replace the included skill with your team's own accounts-payable or contract-review policy, and use your preferred sandbox provider when you deploy the application. Keep approval in your application: the agent can recommend a decision, but it should never make payments or accept contracts on its own.


## Run this notebook

Use a Jupyter Python kernel (Python 3.11 or later) on macOS or Linux in a local clone of the [Cookbook repository](https://github.com/openai/openai-cookbook). The application itself uses Python 3.14; `uv run` installs the dependencies declared in `main.py` and selects that interpreter.

The terminal commands above run the checked-in application. The notebook instead builds a separate copy inside an ignored `tmp_` workspace under your Cookbook checkout. Each `%%writefile` cell contains actual application source. Run these cells in order: the first cell for a module creates its file, and later cells append to it. Python definitions are executed by the application when you launch it.

The setup cell copies only the listed supporting fixtures, manifests, and policy files. It creates a fresh `.env` from the example template without copying your existing credentials. Configure the printed `.env` path before the optional launch step. Rerunning setup creates a new workspace; keep the previous workspace if you need its reports or memory.

Default execution builds and checks the files locally. Docker builds and live API calls require the explicit flags in the launch section.


In [ ]:
from pathlib import Path
import shutil
import tempfile

if globals().get("application_process") is not None and application_process.poll() is None:
    raise RuntimeError("Stop the running application before creating a new workspace.")
application_process = None

# Start Jupyter anywhere inside the Cookbook checkout.
working_directory = Path.cwd().resolve()
cookbook_root = next(
    (path for path in [working_directory, *working_directory.parents]
     if (path / "examples/agents_api/apps/document_review/main.py").is_file()),
    None,
)
if cookbook_root is None:
    raise FileNotFoundError("Clone openai/openai-cookbook and start Jupyter inside it.")

notebook_root = Path(tempfile.mkdtemp(prefix="tmp_agents_document_review_", dir=cookbook_root))
application_dir = notebook_root / "examples/agents_api/apps/document_review"
application_dir.mkdir(parents=True)
for package in [notebook_root / "examples", notebook_root / "examples/agents_api",
                notebook_root / "examples/agents_api/apps", application_dir]:
    (package / "__init__.py").touch()

support_paths = [
    ".env.example",
    "sample_documents/contract.txt",
    "sample_documents/invoice.txt",
    "skills/expense-review-policy/SKILL.md"
]
source_dir = cookbook_root / "examples/agents_api/apps/document_review"
for relative_path in support_paths:
    destination = application_dir / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_dir / relative_path, destination)
shutil.copy2(application_dir / ".env.example", application_dir / ".env")
shutil.copytree(
    cookbook_root / "examples/agents_api/sandboxes/application_managed/docker",
    notebook_root / "examples/agents_api/sandboxes/application_managed/docker",
)
print(f"Application workspace: {application_dir}")
print(f"Configure credentials in: {application_dir / '.env'}")


## Implementation walkthrough

This walkthrough covers mounted documents, reusable policy skills, specialist subagents, structured review artifacts, and human approvals.

Follow the setup instructions above, then build the application with the code cells below.


### 1. Set up

Clone the repository, copy the example's environment template, and build the Docker image that connects an isolated workspace to the Agents API.


### 2. Prepare the document folders

Separate source documents from generated artifacts. The input mount is read-only; the output mount is writable and preserves reports on the host. Use an empty output folder for each batch.

With Docker Desktop, keep both folders under your home directory so the sandbox can access them.


### 3. Add a reusable review skill

Keep your accounts-payable rules in a standard `SKILL.md` file. The included `expense-review-policy` skill defines invoice checks, contract risks, decision statuses, and required report fields.

The skill is mounted at runtime, so updating the policy does not require rebuilding the Docker image.

Read the implementation in [skills/expense-review-policy/SKILL.md](https://github.com/openai/openai-cookbook/blob/main/examples/agents_api/apps/document_review/skills/expense-review-policy/SKILL.md).


### 4. Create a multi-agent review session

Enable multi-agent execution, register the policy skill's capability directory, and tell the coordinating `gpt-5.6-luna` agent to have each specialist apply the discovered skill.


### 5. Mount the documents and policy skill

Start the sandbox with separate document, policy, and artifact mounts. Documents and the reusable skill remain read-only, while specialists write reports into the output directory.

Pass the separate restricted executor key as `CODEX_API_KEY` at runtime. Do not bake credentials into your image or print them in application logs.


### 6. Delegate the document reviews

Ask the coordinator to delegate the batch across specialist subagents. Each specialist discovers and applies the mounted policy before inspecting its document and writing a report.


### 7. Inspect retained command activity

Export retained commands and their turn IDs from the coordinator and each specialist before deleting the session. A null subagent ID identifies the coordinator; specialist commands come from their own retained item histories.

This is not a complete audit of specialist work. Command text can contain document content; protect the activity file like the reports.


### 8. Collect the review artifacts

Read each specialist's report from the mounted output directory, then load the coordinator's consolidated summary. Mark every document as awaiting human approval.


### 9. Inspect the review artifact

Each document gets its own report. For the invoice, the reviewing specialist records the arithmetic error, missing purchase order, and suspicious payment change.


### 10. Leave approval with a person

Inspect the generated reports before approving any invoice or contract. The agent identifies risks and recommendations but never executes an approval.


### 11. Clean up the session

Stop the temporary sandbox and delete the Agents API session. The generated reports remain in the mounted output folder after the container exits.


### 12. Run the sample batch

Review the included invoice and contract together. Specialists inspect both documents and leave individual reports plus a consolidated summary in `review-output`.


## Build the application

The following cells include every application module. Run all cells for each file before launching. The inline dependency declaration in `main.py` installs the current OpenAI SDK and the application libraries through `uv`.


### sandbox.py

Mount source documents and policy skills read-only, and give the executor a separate writable output directory.


In [ ]:
%%writefile "{application_dir}/sandbox.py"
"""Mount the input documents, output artifacts, and policy skill."""

from __future__ import annotations

import os
from pathlib import Path

import docker
from docker.models.containers import Container

EXAMPLE_DIR = Path(__file__).resolve().parent
SKILLS_DIRECTORY = EXAMPLE_DIR / "skills"


def start_executor(
    input_directory: Path, output_directory: Path, environment_id: str, remote_url: str
) -> Container:
    return docker.from_env().containers.run(
        os.environ.get("AGENTS_SANDBOX_IMAGE", "agent-api-sandbox:latest"),
        [
            "codex",
            "exec-server",
            "--remote",
            remote_url,
            "--environment-id",
            environment_id,
        ],
        environment={"CODEX_API_KEY": os.environ["OPENAI_EXECUTOR_API_KEY"]},
        volumes={
            str(input_directory.resolve()): {"bind": "/workspace/input", "mode": "ro"},
            str(output_directory.resolve()): {
                "bind": "/workspace/output",
                "mode": "rw",
            },
            str(SKILLS_DIRECTORY): {"bind": "/workspace/skills", "mode": "ro"},
        },
        detach=True,
        auto_remove=True,
    )


### agent.py

Delegate documents to specialists, export retained command activity, validate report arithmetic, and clean up the sandbox and session.


In [ ]:
%%writefile "{application_dir}/agent.py"
"""Delegate document reviews and export their reports."""

from __future__ import annotations

import asyncio
import json
import logging
import os
import sys
from decimal import Decimal, InvalidOperation
from pathlib import Path
from typing import Any

from docker.errors import NotFound as ContainerNotFound
from docker.models.containers import Container
from openai import AsyncOpenAI, NotFoundError

from .sandbox import SKILLS_DIRECTORY, start_executor

logger = logging.getLogger(__name__)
INSTRUCTIONS = """\
You coordinate bulk invoice and contract reviews.
Before inspecting any document, spawn specialist subagents and assign them the files in
/workspace/input. Assign one document to each specialist when possible, or several for
larger batches. Do not review source documents yourself.

Each specialist must discover and apply $expense-review-policy before inspecting its
assigned documents. Follow the skill's arithmetic checks, decision rules, and required
report fields. Each specialist writes only its /workspace/output/<document-stem>.json,
including the extracted line items and calculation for invoices. Specialists must not
write summary.json. Check their arithmetic and document coverage before summarizing.

Wait for every specialist to finish, then write /workspace/output/summary.json with
document_count, reviews, and recommendation. Never approve a payment or sign a contract.
"""


async def review_activity(client: AsyncOpenAI, session_id: str) -> list[dict[str, Any]]:
    """Export sandbox commands available in retained item history."""
    commands: list[dict[str, Any]] = []
    subagent_ids: list[str | None] = [None]
    subagent_ids.extend(
        [
            subagent.id
            async for subagent in client.beta.agents.sessions.subagents.list(session_id)
        ]
    )
    for subagent_id in subagent_ids:
        items = (
            client.beta.agents.sessions.items.list(session_id, limit=100, order="asc")
            if subagent_id is None
            else client.beta.agents.sessions.subagents.items.list(
                subagent_id, session_id=session_id, limit=100, order="asc"
            )
        )
        async for item in items:
            if item.type == "command_execution":
                commands.append(
                    {
                        "item_id": item.id,
                        "turn_id": item.turn_id,
                        "subagent_id": subagent_id,
                        "command": item.command,
                        "status": item.status,
                    }
                )
    return commands


def validate_review(report: dict[str, Any], document: str) -> None:
    """Reject incomplete reports and inconsistent invoice arithmetic."""
    if (
        report.get("document") != document
        or not report.get("policy_id")
        or report.get("document_type") not in {"invoice", "contract"}
        or report.get("decision")
        not in {"needs_info", "escalated", "ready_for_approval"}
    ):
        raise ValueError(f"{document}: incomplete policy review")
    if report["document_type"] != "invoice":
        return
    try:
        calculation = report["calculation"]
        line_items = calculation["line_items"]
        if not line_items:
            raise ValueError("No invoice line items were extracted")
        amounts = [
            Decimal(str(item["quantity"])) * Decimal(str(item["unit_price"]))
            for item in line_items
        ]
        shipping = Decimal(str(calculation["shipping"]))
        stated = Decimal(str(report["amount"]))
        calculated = Decimal(str(calculation["calculated_total"]))
        difference = Decimal(str(calculation["difference"]))
        if not all(
            value.is_finite()
            for value in [*amounts, shipping, stated, calculated, difference]
        ):
            raise ValueError("Invoice amounts must be finite")
        if sum(amounts) + shipping != calculated or stated - calculated != difference:
            raise ValueError("Invoice totals do not match the extracted line items")
    except (KeyError, TypeError, ValueError, InvalidOperation) as error:
        raise ValueError(f"{document}: invalid invoice calculation: {error}") from error




Continue `agent.py`: `review_documents`. This cell appends to the same file.


In [ ]:
%%writefile -a "{application_dir}/agent.py"
async def review_documents(
    input_directory: Path, output_directory: Path
) -> dict[str, Any]:
    documents = sorted(path for path in input_directory.iterdir() if path.is_file())
    if not documents:
        raise ValueError("The input directory does not contain any documents.")
    if len({document.stem for document in documents}) != len(documents):
        raise ValueError(
            "Document filenames must have unique stems for their JSON reports."
        )
    if any(document.stem == "summary" for document in documents):
        raise ValueError(
            "Rename summary.*; summary.json is reserved for the batch report."
        )
    if any(document.stem == "review-activity" for document in documents):
        raise ValueError(
            "Rename review-activity.*; that name is reserved for command attribution."
        )
    output_directory.mkdir(parents=True, exist_ok=True)
    if any(output_directory.iterdir()):
        raise ValueError(
            "Choose an empty output directory so earlier reports cannot be reused."
        )
    logger.info("Documents: %s", ", ".join(document.name for document in documents))
    logger.info(
        "Input mount: %s -> /workspace/input (read-only)", input_directory.resolve()
    )
    logger.info("Output mount: %s -> /workspace/output", output_directory.resolve())
    logger.info("Skills mount: %s -> /workspace/skills (read-only)", SKILLS_DIRECTORY)

    container: Container | None = None
    async with AsyncOpenAI() as client:
        session = await client.beta.agents.sessions.create(
            agent={
                "model": os.environ.get("OPENAI_MODEL", "gpt-5.6-luna"),
                "instructions": INSTRUCTIONS,
                "reasoning": {"effort": "high"},
                "multi_agent": {
                    "enabled": True,
                    "max_concurrent_subagents": min(len(documents), 8),
                },
            },
            environment={
                "type": "self_hosted",
                "workspace_directory": "/workspace",
                "capability_directories": ["/workspace/skills"],
            },
        )
        logger.info("Session created: %s", session.id)

        try:
            environment = session.environment
            if environment.type != "self_hosted":
                raise RuntimeError("Expected a self-hosted execution environment.")
            logger.info("Environment: %s", environment.id)
            logger.info("Starting document sandbox.")
            container = await asyncio.to_thread(
                start_executor,
                input_directory,
                output_directory,
                environment.id,
                environment.remote_url,
            )
            logger.info("Sandbox started: %s", container.short_id)
            parts: list[str] = []
            subagents: set[str] = set()
            prompt = f"""\
Review all {len(documents)} documents in /workspace/input.
First spawn specialist subagents and assign the documents to them.
Each specialist must apply $expense-review-policy and include its policy_id and decision.
Wait for every review, write one JSON report per document and /workspace/output/summary.json,
and summarize the most important findings for the human approver.
"""
            async with client.beta.agents.sessions.stream(
                session.id, input=prompt
            ) as events:
                async for event in events:
                    if (
                        event.type == "agent.session.subagent.created"
                        and event.subagent.id not in subagents
                    ):
                        subagents.add(event.subagent.id)
                        logger.info(
                            "Specialist started: %s/%s", len(subagents), len(documents)
                        )
                    elif (
                        event.type == "agent.session.turn.item.done"
                        and event.item is not None
                    ):
                        item_type = event.item.type
                        if item_type not in {"reasoning", "message"}:
                            logger.info("Sandbox item: %s", item_type)
                    elif event.type == "agent.session.turn.output_text.delta":
                        parts.append(event.delta)
                    elif (
                        event.type == "agent.session.turn.output_text.done"
                        and not parts
                    ):
                        parts.append(event.text)
                    if event.type in {
                        "agent.session.failed",
                        "agent.session.turn.failed",
                        "error",
                    }:
                        raise RuntimeError(f"Document review failed: {event.to_dict()}")
                    if event.type == "agent.session.turn.cancelled":
                        raise RuntimeError(
                            "Document review was cancelled; reports may be incomplete."
                        )

            if len(documents) > 1 and not subagents:
                raise RuntimeError(
                    "The document batch was not delegated to specialist subagents."
                )

            report_path = output_directory / "summary.json"
            if not report_path.exists():
                response = "".join(parts).strip()
                raise RuntimeError(
                    "The agent did not create /workspace/output/summary.json."
                    + (f" Agent response: {response}" if response else "")
                )

            reviews = []
            for document in documents:
                document_report = output_directory / f"{document.stem}.json"
                if not document_report.exists():
                    raise RuntimeError(
                        f"The agent did not create {document_report.name}."
                    )
                report = json.loads(document_report.read_text())
                validate_review(report, document.name)
                reviews.append(
                    {
                        "document": document.name,
                        "report": report,
                        "status": "awaiting_approval",
                    }
                )
                logger.info("Artifact created: %s", document_report)

            activity = await review_activity(client, session.id)
            activity_path = output_directory / "review-activity.json"
            activity_path.write_text(json.dumps(activity, indent=2) + "\n")
            logger.info(
                "Command attribution saved: %s (%s commands)",
                activity_path,
                len(activity),
            )

            return {
                "summary": "".join(parts),
                "report": json.loads(report_path.read_text()),
                "reviews": reviews,
                "status": "awaiting_approval",
                "session_id": session.id,
                "subagents": len(subagents),
                "activity": activity,
                "output_directory": str(output_directory),
            }
        finally:
            original_error = sys.exception()
            cleanup_errors: list[Exception] = []
            try:
                if container is not None:
                    await asyncio.to_thread(container.remove, force=True)
                    logger.info("Sandbox removed: %s", container.short_id)
            except ContainerNotFound:
                pass
            except Exception as error:
                cleanup_errors.append(error)
            try:
                await client.beta.agents.sessions.delete(session.id)
                logger.info("Session deleted: %s", session.id)
            except NotFoundError:
                pass
            except Exception as error:
                cleanup_errors.append(error)
            if original_error is not None:
                for error in cleanup_errors:
                    original_error.add_note(
                        f"Cleanup for session {session.id}: {error}"
                    )
            elif cleanup_errors:
                raise ExceptionGroup(
                    f"Could not clean up session {session.id} and its sandbox",
                    cleanup_errors,
                )


### main.py

Parse batch paths, load credentials, and print the generated reports and review summary.


In [ ]:
%%writefile "{application_dir}/main.py"
# /// script
# requires-python = ">=3.14"
# dependencies = [
#     "openai>=3.13.0",
#     "docker",
#     "python-dotenv",
# ]
# ///

"""Review a mounted document folder and print the resulting artifacts."""

from __future__ import annotations

import argparse
import asyncio
import logging
import sys
from pathlib import Path

from dotenv import load_dotenv

# Support direct execution from any working directory.
if __package__ in {None, ""}:
    sys.path.insert(0, str(Path(__file__).resolve().parents[4]))


from examples.agents_api.apps.document_review.agent import review_documents

EXAMPLE_DIR = Path(__file__).resolve().parent


async def run_batch(input_directory: Path, output_directory: Path) -> None:
    batch = await review_documents(input_directory, output_directory)
    print(batch["summary"])
    print(f"\nPolicy applied: {batch['reviews'][0]['report']['policy_id']}")
    print(f"Documents reviewed: {len(batch['reviews'])}")
    print(f"Subagents created: {batch['subagents']}")
    print(f"Artifacts: {output_directory.resolve()}")
    for review in batch["reviews"]:
        print(f"  - {review['document']}: {Path(review['document']).stem}.json")
    print("  - summary.json")
    print("  - review-activity.json (retained commands and turn IDs)")
    print("\nStatus: awaiting human approval")


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Review document batches in an isolated sandbox."
    )
    parser.add_argument(
        "--input",
        type=Path,
        required=True,
        metavar="DIRECTORY",
        help="Documents to review.",
    )
    parser.add_argument(
        "--output",
        type=Path,
        default=Path("review-output"),
        help="Output artifact directory.",
    )
    args = parser.parse_args()
    load_dotenv(EXAMPLE_DIR / ".env")
    logging.basicConfig(level=logging.INFO, format="%(message)s")
    logging.getLogger("httpx").setLevel(logging.WARNING)
    asyncio.run(run_batch(args.input, args.output))


if __name__ == "__main__":
    main()


## Check the generated files

Compile all generated modules without importing them or contacting external services. This catches syntax errors before you launch the application.


In [ ]:
import py_compile

modules = ["sandbox.py", "agent.py", "main.py"]
for filename in modules:
    py_compile.compile(str(application_dir / filename), doraise=True)
print(f"Compiled {len(modules)} application modules.")


## Launch the application (optional)

Edit the generated `.env` file with the credentials listed above. Install `uv` and, for sandbox applications, start Docker. The following cells are disabled by default. Enabling them may incur API usage and connect to the configured services.

Use the generated workspace for every path below. For a hosted deployment, package the generated application files and supply credentials through your deployment's secret configuration.


In [ ]:
import subprocess

BUILD_SANDBOX = False
if BUILD_SANDBOX:
    subprocess.run(
        ["docker", "build", "-t", "agent-api-sandbox:latest", str(notebook_root / "examples/agents_api/sandboxes/application_managed/docker")],
        cwd=notebook_root,
        check=True,
    )


The launch arguments review the included invoice and contract in one batch. The process writes to `application.log` in the generated workspace. Inspect that file for errors and progress; the reports remain in `review-output` after the batch finishes.


In [ ]:
import os
import subprocess

RUN_APPLICATION = False
application_arguments = ["--input", str(application_dir / "sample_documents"), "--output", str(application_dir / "review-output")]
if RUN_APPLICATION:
    if application_process is not None and application_process.poll() is None:
        raise RuntimeError("Stop the previous application before launching again.")
    with (application_dir / "application.log").open("w") as application_log:
        application_process = subprocess.Popen(
            ["uv", "run", str(application_dir / "main.py"), *application_arguments],
            cwd=notebook_root,
            env={key: value for key, value in os.environ.items() if key != "VIRTUAL_ENV"},
            start_new_session=True,
            stdout=application_log,
            stderr=subprocess.STDOUT,
        )
    print(f"Process started: {application_process.pid}")
    print(f"Progress log: {application_dir / 'application.log'}")


### Stop a running application

Set `STOP_APPLICATION = True` after you finish. Interrupt the application process group so the application's shutdown handlers can close sessions and remove containers. Batch commands normally exit on their own. A timeout means shutdown is still in progress; inspect the log before taking further action.


In [ ]:
import os
import signal

STOP_APPLICATION = False
if STOP_APPLICATION and application_process is not None:
    if application_process.poll() is None:
        os.killpg(os.getpgid(application_process.pid), signal.SIGINT)
        application_process.wait(timeout=30)
    print(f"Application exited with status {application_process.returncode}.")


The generated workspace remains available for reports and memory. Remove it manually after stopping the application and saving any files you need. Do not rerun the launch cell while the previous process is running.


## Example result

Two specialist subagents discover the mounted policy, review the batch, write individual artifacts, and leave every approval with a human reviewer.

The following illustrates a possible result; model-generated findings depend on the inputs and connected sources.

```text
Policy applied: AP-104
Documents reviewed: 2
Subagents created: 2

review-output/
  invoice.json    $900 overcharge; missing PO; changed bank details
  contract.json   Automatic renewal; unlimited liability; data sharing
  summary.json    Consolidated findings and recommended next steps
  review-activity.json    Command history with specialist IDs

Status: awaiting human approval
```


## Next steps

- Replace the sample skill with your team's accounts-payable or contract-review policy.
- Replace the local Docker container with your preferred isolated sandbox provider.
- Add application tools for vendor records, purchase orders, contract policies, or payment verification.
- Authenticate reviewers and persist every decision in your existing approval and audit system.


## Related documentation

- [Sandbox providers](https://developers.openai.com/api/docs/guides/agents-api/environments/self-hosted#sandbox-providers): Choose a local or hosted provider for isolated document-review workspaces.


## Files

- [main.py](https://github.com/openai/openai-cookbook/blob/main/examples/agents_api/apps/document_review/main.py): Command-line arguments and the batch summary.
- [agent.py](https://github.com/openai/openai-cookbook/blob/main/examples/agents_api/apps/document_review/agent.py): Specialist reviews, report validation, and activity export.
- [sandbox.py](https://github.com/openai/openai-cookbook/blob/main/examples/agents_api/apps/document_review/sandbox.py): Document, artifact, and skill mounts.
